# 10 · The second SLE study (GSE135779, single-cell as pseudobulk) against JDM and the interferonopathies, and against the first SLE study

**Run order:** after 05, 07 and 08. Kernel: **R (endotypes)**. The first run downloads 1.3 GB and takes 15 to 30 minutes; later runs reuse the saved files.

## The question
Is the SLE signal seen in notebook 08 (GSE65391, whole-blood array) also seen in a second, independent SLE study, and how does this second study compare with your diseases and JDM?

## What this study is (Nehar-Belaid D et al. *Nat Immunol* 2020;21:1094-1106)
10x single-cell RNA sequencing of **PBMC** (blood cells without red cells and most neutrophils), 33 children with SLE and 11 healthy children (plus adults, not used). Raw reads are controlled access; the per-donor count matrices are public on GEO. Following Anne's notebooks 20 to 22, the counts of all cells of one child are **summed per gene** into one column per child ("pseudobulk"), so each child becomes one bulk-like sample. No cell types are used.

## Why it is done this way
Same rule as everywhere: this study is **not pooled** with your RNA-seq. SLE is compared with its **own healthy children**, and the resulting fold-change lists are put side by side by gene symbol. Two cautions apply here that did not apply to 08:
- **PBMC is not whole blood.** Your data and GSE65391 are whole blood, which includes neutrophils; PBMC does not. A gene group driven by neutrophils (for example the neutrophil-granule genes) simply cannot be seen in this study.
- **Single-cell counts are sparse**, and pseudobulk sums over several thousand cells per child. Fold changes can differ in size from bulk RNA-seq; compare rank and direction.

## What this notebook does, in order
1. Downloads the GEO files (skipped if present) and checks the by-hand supplement.
2. Metadata: GEO fields joined to the paper's Supplementary Table 1b by donor name.
3. Pseudobulk: one count column per child.
4. **SLE vs healthy**, limma-voom with the study's sequencing **batch** in the model, as Anne's notebook 48.
5. Comparison with every disease of 05, with JDM (07), and with the **first SLE study** (08), each with and without the 28 interferon genes.
6. Interferon score.

**Needs by hand:** the supplement `41590_2020_743_MOESM3_ESM.xlsx` at `/home/jovyan/session_data/Attempt0/data/GSE135779/` (download from the Nature article page, Supplementary Tables 1 to 4).
**Reads:** `rnaseq_out_v2/differential_expression/de_*_vs_HC.csv` (05), `jdm_comparison/de_JDM_vs_HC_jdm_lab.csv` (07), `sle_comparison/de_SLE_vs_healthy_GSE65391.csv` (08).
**Writes:** `data/run_artifacts/GSE135779/metadata.rds` and `pseudobulk_counts.rds` (reusable), and CSV in `rnaseq_out_v2/sle2_comparison/`: `de_SLE_vs_healthy_GSE135779.csv`, `comparison_sle2_vs_others.csv`, `compare_SLE2_vs_<disease>.csv`, `ifn_score_sle2.csv`, `PROCESSING_LOG_sle2.md`.

---
# Part 1 · Settings
| variable | default | meaning |
|---|---|---|
| `ALPHA` | `0.05` | adjusted p-value cut-off for "top genes" |
| `LFC`, `JDM_LFC`, `SLE1_LFC` | `1`, `0.5`, `0.5` | fold-change cut-offs for your RNA-seq diseases, JDM, and the first SLE study (as in 07 and 08) |
| `SLE_LFC` | `0.5` | cut-off for this study |
| `SKIP_DISEASES` | `AGS-like`, `CANDLE-like` | groups of 2 to 3 samples, left out; `character(0)` includes them |

In [ ]:
RESULTS <- "/home/jovyan/session_data/Attempt0/endotype_results"
IN_DIR  <- file.path(RESULTS, "rnaseq_out_v2")
DE_DIR  <- file.path(IN_DIR, "differential_expression")
JDM_DE  <- file.path(IN_DIR, "jdm_comparison", "de_JDM_vs_HC_jdm_lab.csv")
SLE1_DE <- file.path(IN_DIR, "sle_comparison", "de_SLE_vs_healthy_GSE65391.csv")
OUT_DIR <- file.path(IN_DIR, "sle2_comparison")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)
GENE_SET_DIR <- "/home/jovyan/session_data/endotype-transcripts-007/genes"

ALPHA <- 0.05; LFC <- 1; JDM_LFC <- 0.5; SLE1_LFC <- 0.5; SLE_LFC <- 0.5
SKIP_DISEASES <- c("AGS-like", "CANDLE-like")
G <- "GSE135779"

source("/home/jovyan/session_data/Attempt0/src/paths.R")      # gives raw(), art(), SEED
suppressMessages({library(limma); library(edgeR); library(Matrix); library(GEOquery); library(readxl)})

SUPP <- raw(G, "41590_2020_743_MOESM3_ESM.xlsx")
if (!file.exists(SUPP)) stop("The supplement is not at:\n  ", SUPP, "\nDownload Supplementary Tables 1-4 from the Nature article page by hand and put it there (mkdir -p its folder first).")
if (!identical(rawToChar(readBin(SUPP, "raw", 2)), "PK"))
  stop("The supplement file is not a real Excel file (probably the publisher's check page, a few KB). Download it again in a browser.")
for (f in c(JDM_DE, SLE1_DE)) if (!file.exists(f)) stop("Missing: ", f, " (notebook 07 / 08 must have run)")
nih_files <- list.files(DE_DIR, pattern = "^de_.*_vs_HC\\.csv$", full.names = TRUE)
if (!length(nih_files)) stop("No de_<disease>_vs_HC.csv in ", DE_DIR, ": run notebook 05 first.")
read_set <- function(f) { p <- file.path(GENE_SET_DIR, f); if (!file.exists(p)) return(NULL)
  x <- trimws(readLines(p, warn = FALSE)); x[x != "" & !startsWith(x, "#")] }
irg <- read_set("ifn-response-genes-28.txt")
cat("settings OK; supplement found; output folder:", OUT_DIR, "\n")

---
# Part 2 · Download
Three files from GEO (the per-donor matrices as one 1.3 GB `.tar`, the gene list, the series matrix), skipped when already present. Then the matrices are unpacked. Expect **56** matrices (one per GEO sample; adults included at this stage).

In [ ]:
options(timeout = 7200)
base <- "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE135nnn/GSE135779"
files <- c(tar = raw(G, "GSE135779_RAW.tar"), genes = raw(G, "GSE135779_genes.tsv.gz"), series = raw(G, "GSE135779_series_matrix.txt.gz"))
urls  <- c(tar = file.path(base, "suppl/GSE135779_RAW.tar"), genes = file.path(base, "suppl/GSE135779_genes.tsv.gz"),
           series = file.path(base, "matrix/GSE135779_series_matrix.txt.gz"))
for (f in names(files)) if (!file.exists(files[[f]])) { cat("downloading", basename(files[[f]]), "...\n"); download.file(urls[[f]], files[[f]], mode = "wb") }
print(data.frame(file = basename(files), MB = round(file.size(files) / 1e6, 2)))
rawdir <- raw(G, "raw")
if (!length(list.files(rawdir, pattern = "matrix.mtx.gz$"))) { dir.create(rawdir, showWarnings = FALSE); untar(files[["tar"]], exdir = rawdir) }
c(matrices = length(list.files(rawdir, pattern = "matrix.mtx.gz$")), barcode_files = length(list.files(rawdir, pattern = "barcodes.tsv.gz$")))

---
# Part 3 · Metadata
GEO gives, per sample: age, age group (Children or Adults) and group (SLE or HD, healthy donor). Only **children** are kept. The clinical data (including the study's sequencing `Batch`) come from the paper's Supplementary Table 1b, joined to GEO by donor name (`cSLE1`, `cHD1`, ...). Checks printed: all children must match a row, no name may appear twice, and age and group must agree between GEO and the table.

In [ ]:
pd <- pData(suppressMessages(getGEO(filename = files[["series"]], getGPL = FALSE)))
meta <- data.frame(sample = rownames(pd), donor = sub(" \\[.*", "", pd$title), library = sub(".*\\[(.*)\\]", "\\1", pd$title),
                   age = as.numeric(pd[["age:ch1"]]), age_group = pd[["age group:ch1"]],
                   sle = as.numeric(pd[["groups:ch1"]] == "SLE"), row.names = rownames(pd))
print(table(age_group = meta$age_group, sle = meta$sle))
meta <- meta[meta$age_group == "Children", ]
print(c(children = nrow(meta), SLE = sum(meta$sle), healthy = sum(meta$sle == 0)))

st <- as.data.frame(read_excel(SUPP, sheet = "ST1b-Clinical information ", skip = 3, na = c("ND", "NA", "")))
stopifnot(all(meta$donor %in% st$Names), !anyDuplicated(st$Names))
tb <- st[match(meta$donor, st$Names), ]
print(c(age_agrees = sum(as.numeric(tb$Age) == meta$age, na.rm = TRUE), group_agrees = sum((tb$Groups == "cSLE") == (meta$sle == 1)), children = nrow(meta)))
meta <- cbind(meta, tb[, setdiff(names(tb), c("Names", "Age", "Groups"))])
if (!"Batch" %in% names(meta)) stop("No 'Batch' column in the clinical table. Columns found: ", paste(names(meta), collapse = ", "))
print(table(group = ifelse(meta$sle == 1, "SLE", "healthy"), batch = meta$Batch))
saveRDS(meta, art(G, "metadata.rds"))

---
# Part 4 · Pseudobulk
For each child, the UMI counts of every cell in that child's matrix are summed per gene (no cell-type labels). All matrices share the gene list `GSE135779_genes.tsv.gz` (Ensembl ID and symbol). Reading 44 matrices takes several minutes. The result is saved so later runs skip this part.

In [ ]:
pb_file <- art(G, "pseudobulk_counts.rds")
if (file.exists(pb_file)) {
  pb <- readRDS(pb_file); cat("reusing", pb_file, "\n")
} else {
  genes <- read.delim(files[["genes"]], header = FALSE, col.names = c("ensembl", "symbol"))
  mfiles <- list.files(rawdir, pattern = "matrix.mtx.gz$", full.names = TRUE)
  mfiles <- setNames(mfiles, sub("_.*", "", basename(mfiles)))
  stopifnot(all(rownames(meta) %in% names(mfiles)))
  counts <- matrix(0, nrow(genes), nrow(meta), dimnames = list(genes$ensembl, rownames(meta)))
  cells <- setNames(integer(nrow(meta)), rownames(meta))
  for (g in rownames(meta)) {
    m <- readMM(gzfile(mfiles[[g]]))
    stopifnot(nrow(m) == nrow(genes))
    cells[g] <- ncol(m); counts[, g] <- rowSums(m)
  }
  stopifnot(all(counts == round(counts)))
  meta$cells <- cells[rownames(meta)]
  pb <- list(counts = counts, genes = genes, meta = meta)
  saveRDS(pb, pb_file)
}
meta <- pb$meta; counts <- pb$counts; genes <- pb$genes
print(summary(data.frame(cells = meta$cells, total_counts_millions = round(colSums(counts) / 1e6, 1))))
c(genes = nrow(counts), children = ncol(counts))

---
# Part 5 · SLE against healthy children (this study's own comparison)
**Model:** limma-voom, `~ group + batch` (the study's sequencing batch, from the clinical table), on genes passing `filterByExpr` with TMM normalisation (Anne's notebook 48; Law CW et al. *Genome Biol* 2014;15:R29). A batch column that cannot be told apart from the group (a batch holding only one group) is dropped and printed. A positive log fold change means higher in SLE.

**Sanity check against Anne's run on the same data:** 15,414 genes pass the filter, 33 SLE and 11 healthy children, 2,222 genes with adjusted p < 0.05. Yours should be close.

In [ ]:
grp <- factor(ifelse(meta$sle == 1, "SLE", "healthy"), levels = c("healthy", "SLE"))
if (anyNA(meta$Batch)) stop(sum(is.na(meta$Batch)), " child(ren) have no Batch in the clinical table; the model needs one for each.")
bat <- factor(meta$Batch)
design <- model.matrix(~ grp + bat)
q <- qr(design)
if (q$rank < ncol(design)) {                    # group is column 2; only batch columns are dropped
  keep_cols <- sort(q$pivot[seq_len(q$rank)])
  cat("batch columns dropped (not separable from the group):", colnames(design)[-keep_cols], "\n")
  design <- design[, keep_cols, drop = FALSE]
}
stopifnot("grpSLE" %in% colnames(design))
y <- DGEList(counts); keep <- filterByExpr(y, design)
y <- calcNormFactors(y[keep, , keep.lib.sizes = FALSE])
fit <- eBayes(lmFit(voom(y, design), design))
tt <- topTable(fit, coef = "grpSLE", number = Inf, sort.by = "P")
tt$ensembl <- rownames(tt); tt$gene_name <- genes$symbol[match(tt$ensembl, genes$ensembl)]
tt <- tt[!is.na(tt$gene_name) & tt$gene_name != "", ]
tt <- tt[order(-tt$AveExpr), ]; tt <- tt[!duplicated(tt$gene_name), ]; tt <- tt[order(tt$P.Value), ]
write.csv(tt, file.path(OUT_DIR, "de_SLE_vs_healthy_GSE135779.csv"), row.names = FALSE)
res5 <- c(genes_pass_filter = sum(keep), SLE = sum(grp == "SLE"), healthy = sum(grp == "healthy"),
          adj_p_below_0.05 = sum(topTable(fit, coef = "grpSLE", number = Inf)$adj.P.Val < ALPHA),
          top_up = sum(tt$adj.P.Val < ALPHA & tt$logFC >= SLE_LFC), top_down = sum(tt$adj.P.Val < ALPHA & tt$logFC <= -SLE_LFC))
print(res5)

---
# Part 6 · Against each disease, JDM and the first SLE study, gene by gene
For each one, its fold-change list against its own healthy controls (notebook 05 for your diseases, 07 for JDM, 08 for the first SLE study) is matched to this study's SLE-vs-healthy list **by gene symbol**. Columns, as in 07 and 08:
- `rho_all_genes`: Spearman rank correlation of the two fold-change lists (near 1 = same genes move the same way, near 0 = unrelated, negative = opposite);
- `rho_without_ifn`: the same after removing the 28 interferon genes (a weak test: 28 genes cannot move a correlation over thousands of genes; the shared top-gene counts are more informative);
- `top_in_both` and `top_in_both_same_direction`: genes passing the cut-off on both sides, and how many go the same way.

**The row to watch is `SLE_GSE65391`**, the first SLE study: it is the same disease in a different technology, tissue and cohort. Anne's own check between her two SLE studies (notebook 48, over 9,971 shared genes) found Spearman 0.42. Gene symbols in the single-cell gene list can be older than the current ones, so a few genes will not match.

In [ ]:
others <- list()
for (f in nih_files) {
  dis <- sub("_vs_HC$", "", sub("^de_", "", sub("\\.csv$", "", basename(f))))
  if (dis %in% SKIP_DISEASES) next
  n <- read.csv(f); n <- n[order(-n$AveExpr), ]; n <- n[!duplicated(n$gene_name), ]
  others[[dis]] <- list(d = n, lfc = LFC)
}
jd <- read.csv(JDM_DE); others[["JDM"]] <- list(d = jd[!duplicated(jd$gene_name), ], lfc = JDM_LFC)
s1 <- read.csv(SLE1_DE); others[["SLE_GSE65391"]] <- list(d = s1[!duplicated(s1$gene_name), ], lfc = SLE1_LFC)

sle2 <- tt[, c("gene_name", "logFC", "adj.P.Val")]
rows <- lapply(names(others), function(dis) {
  o <- others[[dis]]
  m <- merge(sle2, o$d[, c("gene_name", "logFC", "adj.P.Val")], by = "gene_name", suffixes = c("_sle2", "_other"))
  m$ifn_gene <- if (is.null(irg)) FALSE else m$gene_name %in% irg
  sn <- m$adj.P.Val_sle2 < ALPHA & abs(m$logFC_sle2) >= SLE_LFC
  so <- m$adj.P.Val_other < ALPHA & abs(m$logFC_other) >= o$lfc
  both <- sn & so; same <- sign(m$logFC_sle2) == sign(m$logFC_other)
  m0 <- m[!m$ifn_gene, ]
  write.csv(m, file.path(OUT_DIR, sprintf("compare_SLE2_vs_%s.csv", dis)), row.names = FALSE)
  data.frame(disease = dis, genes_compared = nrow(m), ifn_genes_in_comparison = sum(m$ifn_gene),
             rho_all_genes = round(cor(m$logFC_sle2, m$logFC_other, method = "spearman"), 3),
             rho_without_ifn = round(cor(m0$logFC_sle2, m0$logFC_other, method = "spearman"), 3),
             top_sle2 = sum(sn), top_other = sum(so), top_in_both = sum(both), top_in_both_same_direction = sum(both & same),
             stringsAsFactors = FALSE)
})
cmp <- do.call(rbind, rows); rownames(cmp) <- NULL
cmp$reading <- ifelse(cmp$rho_all_genes <= 0.2, "no similarity to begin with",
               ifelse(cmp$rho_without_ifn > 0.2 & cmp$rho_without_ifn >= 0.5 * cmp$rho_all_genes, "holds without interferon genes", "mostly interferon"))
print(cmp, right = FALSE)
write.csv(cmp, file.path(OUT_DIR, "comparison_sle2_vs_others.csv"), row.names = FALSE)

options(repr.plot.width = 4.5 * length(rows), repr.plot.height = 4.5)
par(mfrow = c(1, length(rows)), mar = c(4, 4, 3, 1))
for (dis in names(others)) {
  mm <- read.csv(file.path(OUT_DIR, sprintf("compare_SLE2_vs_%s.csv", dis)))
  plot(mm$logFC_other, mm$logFC_sle2, pch = 16, cex = 0.3, col = adjustcolor("grey30", 0.35),
       xlab = sprintf("%s vs its healthy (log2 FC)", dis), ylab = "SLE (GSE135779) vs healthy (log2 FC)",
       main = sprintf("%s: rho = %.2f", dis, cmp$rho_all_genes[cmp$disease == dis]), cex.main = 0.9)
  abline(h = 0, v = 0, lty = 2, col = "grey60")
  if (any(mm$ifn_gene)) points(mm$logFC_other[mm$ifn_gene], mm$logFC_sle2[mm$ifn_gene], pch = 16, cex = 0.9, col = "#B2182B")
}

---
# Part 7 · Interferon score
Mean over the 28 interferon response genes of each child's z-score against **this study's healthy children** (log2 counts per million; floor of 0.1 on the spread), as in 06 to 08. Genes are matched by symbol; not all 28 may be in this gene list. The score is not corrected for the study's batch, so read it with that in mind.

In [ ]:
if (is.null(irg)) {
  cat("Gene list not found in", GENE_SET_DIR, "- skipped.\n")
} else {
  lc <- cpm(y, log = TRUE)
  sym <- genes$symbol[match(rownames(lc), genes$ensembl)]
  o <- order(rowMeans(lc), decreasing = TRUE); o <- o[!is.na(sym[o]) & sym[o] != "" & !duplicated(sym[o])]
  L <- lc[o, , drop = FALSE]; rownames(L) <- sym[o]
  g <- intersect(irg, rownames(L))
  cat("interferon genes found:", length(g), "of", length(irg), "\n")
  H <- L[g, grp == "healthy", drop = FALSE]
  Z <- (L[g, , drop = FALSE] - rowMeans(H)) / pmax(apply(H, 1, sd), 0.1)
  sc <- colMeans(Z)
  tab <- data.frame(group = c("HC (GSE135779)", "SLE"), samples = c(sum(grp == "healthy"), sum(grp == "SLE")),
                    median_ifn_score = round(c(median(sc[grp == "healthy"]), median(sc[grp == "SLE"])), 2), genes_used = length(g))
  print(tab, row.names = FALSE)
  write.csv(tab, file.path(OUT_DIR, "ifn_score_sle2.csv"), row.names = FALSE)
  options(repr.plot.width = 5, repr.plot.height = 4.5)
  boxplot(sc ~ grp, ylab = "interferon score (z against this study's healthy children)", xlab = "", col = c("grey80", "#7570B3"))
  abline(h = 0, lty = 2, col = "grey50")
}

---
# Part 8 · Log

In [ ]:
log_lines <- c("# Second SLE study log (notebook 10)", paste0("Generated ", format(Sys.time(), "%Y-%m-%d %H:%M"), "\n"),
               "## Settings", paste0("ALPHA=", ALPHA, "; LFC=", LFC, "; JDM_LFC=", JDM_LFC, "; SLE1_LFC=", SLE1_LFC, "; SLE_LFC=", SLE_LFC,
                                     "; skipped: ", paste(SKIP_DISEASES, collapse = ", ")), "",
               "## SLE vs healthy (GSE135779)", "```", capture.output(print(res5)), "```", "",
               "## Against each disease, JDM and the first SLE study", "```", capture.output(print(cmp, right = FALSE)), "```")
writeLines(log_lines, file.path(OUT_DIR, "PROCESSING_LOG_sle2.md"))
print(list.files(OUT_DIR))

---
## What to check
1. **Part 3:** 44 children (33 SLE, 11 healthy), all matched to the table, age and group agreeing. A batch table where one batch holds only one group is expected (Anne notes batch B1).
2. **Part 5:** close to Anne's numbers (about 15,414 genes pass, 2,222 significant). A very different number means a step above differed.
3. **Part 6, `SLE_GSE65391` row:** a clearly positive correlation (Anne found about 0.42 over shared genes) means the two SLE studies agree. A value near zero would mean the SLE signal does not replicate here, which would be a finding worth discussing.
4. **Part 6, your diseases:** compare with notebook 08's values (SAVI 0.46, UAID 0.44, JDM 0.40, CANDLE 0.37, AGS negative). If this second SLE study agrees on the order, that strengthens it; if it differs, the difference is probably tissue (PBMC against whole blood).
5. **Part 7:** SLE above its healthy children.

## Limits
- **PBMC, not whole blood.** Neutrophil-driven signals cannot appear, and the cell mix differs from your data.
- **Pseudobulk of single-cell data** is sparse; fold-change sizes are not comparable with bulk RNA-seq, only rank and direction.
- **Sample size:** 11 healthy children, one batch holding only one group, no adjustment for age or sex.
- **Gene-name matching** between this gene list (older symbols) and the others is imperfect.

## References
1. Nehar-Belaid D et al. Mapping systemic lupus erythematosus heterogeneity at the single-cell level. *Nat Immunol* 2020;21:1094-1106.
2. Law CW, Chen Y, Shi W, Smyth GK. voom: precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biol* 2014;15:R29.
3. Ritchie ME et al. limma powers differential expression analyses for RNA-sequencing and microarray studies. *Nucleic Acids Res* 2015;43:e47.
4. Kim H et al. *J Interferon Cytokine Res* 2018;38:171-185; de Jesus AA et al. *J Clin Invest* 2020;130:1669-1682 (interferon response genes).
5. Anne Deslattes Mays, endotypes-transcriptomics, notebooks 20 to 22 and 48.